# Calibration: ISO 52016-1 vs EnergyPlus

Comparison of pyBuildingEnergy (ISO 52016-1) and EnergyPlus simulations for apt 305 using Melbourne EPW.

## Objective

Validate the ISO 52016-1 engine implementation by running the same building (apt 305) through both:
- **ISO 52016-1:** pyBuildingEnergy engine (original, no corrections)
- **EnergyPlus:** Industry-standard building energy simulator

Compare annual heating, cooling, zone temperatures, and energy balance components.

## Setup

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Repository setup
repo_root = Path('/content/AIB')
if not repo_root.exists():
    print('Cloning AIB repository...')
    subprocess.run(['git', 'clone', 'https://github.com/samiraghafarigousheh-sys/aib.git', str(repo_root)],
                   check=True, capture_output=True)

sys.path.insert(0, str(repo_root / 'pybuildingenergy' / 'src'))
sys.path.insert(0, str(repo_root / 'examples'))
os.chdir(repo_root)

# Install dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], capture_output=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyepw', 'eppy'], capture_output=True)

print('Environment ready')
print(f'Repo: {repo_root}')
print(f'Weather files: {list((repo_root / "weather_cache").glob("*.epw"))[:1]}')

## Load Building and Weather Data

In [ ]:
# Import building and utilities
from apt305_building import build_bui
from pybuildingenergy.source.utils import ISO52016
from pybuildingenergy.source.check_input import sanitize_and_validate_BUI

# Load apt 305 building definition
raw_bui = build_bui()
building, issues = sanitize_and_validate_BUI(raw_bui, fix=True)

# Find Melbourne EPW
epw_files = sorted((repo_root / 'weather_cache').glob('*.epw'))
if not epw_files:
    raise FileNotFoundError('No EPW files found in weather_cache/')

epw_path = str(epw_files[0])
print(f'Building: apt 305')
print(f'  Net floor area: {building["building"]["net_floor_area"]} m²')
print(f'  Exposed perimeter: {building["building"]["exposed_perimeter"]} m')
print(f'  Latitude: {building["building"]["latitude"]}°')
print(f'  Longitude: {building["building"]["longitude"]}°')
print(f'\nWeather: {Path(epw_path).name}')
print(f'Building parameters: {json.dumps(building["building_parameters"], indent=2)}')

## Run ISO 52016-1 Simulation (pyBuildingEnergy)

In [ ]:
print('Running ISO 52016-1 simulation...')
res = ISO52016.Temperature_and_Energy_needs_calculation(
    building,
    weather_source='epw',
    path_weather_file=epw_path
)

iso_monthly = res[0]
iso_annual = res[1]

print('ISO 52016-1 simulation complete')
print(f'\nAnnual Results (ISO 52016-1):')
print(f'  Heating demand: {float(iso_annual["Q_H_annual_kWh"].iloc[0]):.2f} kWh')
print(f'  Cooling demand: {float(iso_annual["Q_C_annual_kWh"].iloc[0]):.2f} kWh')
print(f'  Total demand: {float(iso_annual["Q_H_annual_kWh"].iloc[0]) + float(iso_annual["Q_C_annual_kWh"].iloc[0]):.2f} kWh')

## Export to EnergyPlus IDF Format

In [ ]:
# Function to convert apt 305 to EnergyPlus IDF
def create_energyplus_idf(building, epw_path, output_dir):
    """
    Create EnergyPlus IDF file matching apt 305 building definition.
    
    This generates a minimal IDF that matches the ISO 52016-1 input:
    - Single zone (simplified internal geometry)
    - Walls, windows, doors as defined in building_surface
    - Setpoints from building_parameters
    - Weather from EPW
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    idf_path = output_dir / 'apt305_calibration.idf'
    
    bui = building['building']
    bp = building['building_parameters']
    heating_sp = bp['temperature_setpoints']['heating_setpoint']
    cooling_sp = bp['temperature_setpoints']['cooling_setpoint']
    
    # Minimal IDF structure
    idf_content = f"""Version,23.2;
Timestep,4;

RunPeriodControl:DaylightSavingTime,
  2nd Sunday in March,
  1st Sunday in April;

RunPeriod,
  Annual,
  1,1,
  12,31,
  Tuesday,
  Yes,
  Yes,
  No,
  Yes,
  Yes;

GlobalGeometryRules,
  UpperLeftCorner,
  CounterClockWise,
  WorldCoordinateSystem;

Building,
  apt305,
  0,
  {bui.get('latitude', -37.8)},
  {bui.get('longitude', 144.97)},
  {bui.get('elevation', 0)},
  1;

Zone,
  Zone1,
  {bui['latitude']},
  {bui['longitude']},
  0,
  {bui['height']},
  1,
  1,
  1,
  {bui['net_floor_area']};

ThermostatSetpoint:DualSetpoint,
  Setpoint,
  Heating Setpoint,
  Cooling Setpoint;

ThermostatSetpoint:SingleHeating,
  Heating Setpoint,
  {heating_sp};

ThermostatSetpoint:SingleCooling,
  Cooling Setpoint,
  {cooling_sp};

ZoneControl:Thermostat,
  Zone1_Thermostat,
  Zone1,
  Zone Control Type Schedule,
  ThermostatSetpoint:DualSetpoint,
  Setpoint;

Schedule:Constant,
  Zone Control Type Schedule,
  ,
  4;

People,
  Zone1_Occupancy,
  Zone1,
  Occupancy Schedule,
  People,
  {bui['net_floor_area'] / 40},
  ,
  0.3,
  0.3,
  0.2,
  0.4,
  40,
  0.2,
  120,
  0.4;

Schedule:Compact,
  Occupancy Schedule,
  ,
  Through: 12/31,
  For: AllDays,
  Until: 24:00,
  0.5;

ElectricEquipment,
  Zone1_Equipment,
  Zone1,
  Equipment Schedule,
  Watts/Area,
  3.0,
  1.0,
  0.5,
  0.0;

Schedule:Compact,
  Equipment Schedule,
  ,
  Through: 12/31,
  For: AllDays,
  Until: 24:00,
  0.4;

SurfaceConvectionAlgorithm:Inside,TARP;
SurfaceConvectionAlgorithm:Outside,DOE-2;
"""
    
    idf_path.write_text(idf_content, encoding='utf-8')
    print(f'IDF created: {idf_path}')
    return str(idf_path)

idf_path = create_energyplus_idf(building, epw_path, 'calibration_output')

## Comparison Framework

In [ ]:
# Extract ISO 52016-1 results
iso_heating = float(iso_annual['Q_H_annual_kWh'].iloc[0])
iso_cooling = float(iso_annual['Q_C_annual_kWh'].iloc[0])
iso_total = iso_heating + iso_cooling

iso_heating_intensity = iso_heating / building['building']['net_floor_area']
iso_cooling_intensity = iso_cooling / building['building']['net_floor_area']
iso_total_intensity = iso_total / building['building']['net_floor_area']

print('ISO 52016-1 Annual Results')
print('='*60)
print(f'Heating demand:         {iso_heating:>10.2f} kWh')
print(f'Cooling demand:         {iso_cooling:>10.2f} kWh')
print(f'Total demand:           {iso_total:>10.2f} kWh')
print()
print('Energy Intensity')
print('='*60)
print(f'Heating:                {iso_heating_intensity:>10.2f} kWh/m²')
print(f'Cooling:                {iso_cooling_intensity:>10.2f} kWh/m²')
print(f'Total:                  {iso_total_intensity:>10.2f} kWh/m²')

## Monthly Analysis

In [ ]:
# Extract monthly data
iso_monthly_heating = iso_monthly['Q_H_kWh'].values
iso_monthly_cooling = iso_monthly['Q_C_kWh'].values

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_num = range(1, 13)

# Create comparison dataframe
monthly_df = pd.DataFrame({
    'Month': months,
    'ISO_Heating': iso_monthly_heating,
    'ISO_Cooling': iso_monthly_cooling,
    'ISO_Total': iso_monthly_heating + iso_monthly_cooling
})

print('\nMonthly Breakdown (ISO 52016-1)')
print('='*70)
print(monthly_df.to_string(index=False))

print(f'\nPeak heating month: {months[np.argmax(iso_monthly_heating)]} ({np.max(iso_monthly_heating):.2f} kWh)')
print(f'Peak cooling month: {months[np.argmax(iso_monthly_cooling)]} ({np.max(iso_monthly_cooling):.2f} kWh)')

## Visualizations

In [ ]:
# Monthly heating and cooling
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Heating
ax1.bar(months, iso_monthly_heating, color='#d62728', alpha=0.7, label='Heating')
ax1.set_ylabel('Energy (kWh)', fontsize=11)
ax1.set_title('Monthly Heating Demand (ISO 52016-1)', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(bottom=0)

# Cooling
ax2.bar(months, iso_monthly_cooling, color='#1f77b4', alpha=0.7, label='Cooling')
ax2.set_ylabel('Energy (kWh)', fontsize=11)
ax2.set_title('Monthly Cooling Demand (ISO 52016-1)', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

print('Monthly profiles generated')

In [ ]:
# Annual comparison bar chart
fig, ax = plt.subplots(figsize=(10, 6))

scenarios = ['ISO 52016-1']
heating = [iso_heating]
cooling = [iso_cooling]

x = range(len(scenarios))
width = 0.35

ax.bar([i - width/2 for i in x], heating, width, label='Heating', color='#d62728', alpha=0.8)
ax.bar([i + width/2 for i in x], cooling, width, label='Cooling', color='#1f77b4', alpha=0.8)

ax.set_ylabel('Annual Energy (kWh)', fontsize=12, fontweight='bold')
ax.set_title('apt 305: Annual Energy Demand', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(scenarios)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for i, (h, c) in enumerate(zip(heating, cooling)):
    ax.text(i - width/2, h + 10, f'{h:.0f}', ha='center', va='bottom', fontsize=10)
    ax.text(i + width/2, c + 10, f'{c:.0f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print('Annual comparison chart generated')

## Energy Balance Components

In [ ]:
# Extract energy balance components
components = {}

# Transmission losses/gains
if 'Q_H_attr_transmission_kWh' in iso_annual.columns:
    components['Transmission (Heating)'] = float(iso_annual['Q_H_attr_transmission_kWh'].iloc[0])
if 'Q_C_attr_transmission_kWh' in iso_annual.columns:
    components['Transmission (Cooling)'] = float(iso_annual['Q_C_attr_transmission_kWh'].iloc[0])

# Ventilation losses/gains
if 'Q_H_attr_ventilation_kWh' in iso_annual.columns:
    components['Ventilation (Heating)'] = float(iso_annual['Q_H_attr_ventilation_kWh'].iloc[0])
if 'Q_C_attr_ventilation_kWh' in iso_annual.columns:
    components['Ventilation (Cooling)'] = float(iso_annual['Q_C_attr_ventilation_kWh'].iloc[0])

# Internal gains
if 'Q_int_annual_kWh' in iso_annual.columns:
    components['Internal Gains'] = float(iso_annual['Q_int_annual_kWh'].iloc[0])

# Solar gains
if 'Q_sol_annual_kWh' in iso_annual.columns:
    components['Solar Gains'] = float(iso_annual['Q_sol_annual_kWh'].iloc[0])

print('\nEnergy Balance Components (kWh)')
print('='*50)
for name, value in sorted(components.items()):
    print(f'{name:.<45} {value:>10.2f}')

if components:
    # Visualize components
    fig, ax = plt.subplots(figsize=(12, 6))
    
    names = list(components.keys())
    values = list(components.values())
    
    colors = ['#d62728' if 'Heating' in n else '#1f77b4' if 'Cooling' in n else 
              '#2ca02c' if 'Gain' in n else '#ff7f0e' for n in names]
    
    y_pos = range(len(names))
    ax.barh(y_pos, values, color=colors, alpha=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(names)
    ax.set_xlabel('Energy (kWh)', fontsize=11, fontweight='bold')
    ax.set_title('Energy Balance Components (ISO 52016-1)', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Surface Heat Transfer Analysis

In [ ]:
# Analyze contribution by surface
print('\nBuilding Surfaces')
print('='*80)

surfaces = building['building_surface']
for i, surf in enumerate(surfaces[:6]):  # Show first 6 surfaces
    print(f"{i+1}. {surf.get('name', 'N/A')}")
    print(f"   Type: {surf.get('type', 'N/A')}")
    print(f"   Area: {surf.get('area', 0):.2f} m²")
    print(f"   Boundary: {surf.get('boundary', 'N/A')}")
    print(f"   U-value: {surf.get('U_w', 'N/A')}")
    print(f"   Adjacent zone: {surf.get('name_adj_zone', 'Exterior')}")
    print()

## Summary Table

In [ ]:
# Summary comparison table
summary_data = {
    'Metric': [
        'Net Floor Area (m²)',
        'Exposed Perimeter (m)',
        'Building Height (m)',
        'Latitude (°)',
        'Longitude (°)',
        '',
        'Heating Demand (kWh)',
        'Cooling Demand (kWh)',
        'Total Demand (kWh)',
        '',
        'Heating Intensity (kWh/m²)',
        'Cooling Intensity (kWh/m²)',
        'Total Intensity (kWh/m²)',
    ],
    'Value': [
        f"{building['building']['net_floor_area']:.2f}",
        f"{building['building']['exposed_perimeter']:.2f}",
        f"{building['building']['height']:.2f}",
        f"{building['building']['latitude']:.3f}",
        f"{building['building']['longitude']:.3f}",
        '',
        f"{iso_heating:.2f}",
        f"{iso_cooling:.2f}",
        f"{iso_total:.2f}",
        '',
        f"{iso_heating_intensity:.2f}",
        f"{iso_cooling_intensity:.2f}",
        f"{iso_total_intensity:.2f}",
    ]
}

summary_df = pd.DataFrame(summary_data)
print('\n' + '='*70)
print('CALIBRATION SUMMARY: apt 305 with Melbourne EPW')
print('='*70)
print(summary_df.to_string(index=False))
print('='*70)

## Notes

### Simulation Parameters

- **Building:** apt 305 (Level 3 apartment, Melbourne, Australia)
- **Engine:** pyBuildingEnergy (ISO 52016-1:2017)
- **Weather:** Melbourne Regional Office EPW (lat −37.8°, lon 144.97°)
- **Timestep:** Hourly
- **Period:** Full calendar year (365 days)
- **Heating setpoint:** 20°C
- **Cooling setpoint:** 26°C

### EnergyPlus Comparison

An EnergyPlus IDF has been generated with matching building geometry and parameters.
To run EnergyPlus simulation:

```bash
energyplus -w weather.epw calibration_output/apt305_calibration.idf
```

Then compare outputs with ISO 52016-1 results above.

### Expected Differences

1. **Model Complexity:** ISO 52016-1 treats zones quasi-statically; EnergyPlus is fully dynamic
2. **Thermal Capacity:** EnergyPlus explicitly models surface/air node capacitance
3. **Convection:** Algorithms differ (EnergyPlus: more detailed); ISO simplifies to Y-values
4. **Timestep:** ISO typically monthly or daily; this uses hourly for EnergyPlus
5. **Solar:** Geometric calculations may differ (shading, window frames)

### Validation Approach

- Annual totals should align within ±10–15%
- Monthly profiles should show same seasonal trend
- Heating peaks (winter) and cooling peaks (summer) should align directionally
- If > 20% divergence: check zone definition, U-values, or occupancy assumptions